In [1]:
import pandas as pd
import numpy as np
import random
import re

In [2]:
production = pd.read_csv("production_events_cleaned_T9.csv")
quality = pd.read_csv("quality_inspection_cleaned_T6.csv")
sensor = pd.read_csv("sensor_readings_cleaned_T10.csv")
supplier = pd.read_csv("supplier_master.csv")
bom = pd.read_csv("bom_master.csv")
workorder = pd.read_csv("work_orders.csv")
tool = pd.read_csv("tool_master.csv")
calibration = pd.read_csv("calibration_master.csv")
operator = pd.read_csv("operator_master.csv")
warranty = pd.read_csv("warranty_claims.csv")

# Test Case 11

In [3]:
production["Timestamp"] = pd.to_datetime(
    production["Timestamp"],
    errors="coerce"
)

In [4]:
production["Station"] = (
    production["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [5]:
production["Cycle_Time_Sec"] = (
    production.groupby("Station")["Timestamp"]
    .diff()
    .dt.total_seconds()
)


In [6]:
def cycle_status(time):
    if time == 0:
        return "First Record"
    elif time < 30:
        return "Too Fast"
    elif time > 300:
        return "Too Slow"
    else:
        return "Normal"

production["Cycle_Time_Status"] = production["Cycle_Time_Sec"].apply(cycle_status)

In [7]:
print(production[
    [
        "Station",
        "Timestamp",
        "Cycle_Time_Sec",
        "Cycle_Time_Status"
    ]
].head(20))

   Station           Timestamp  Cycle_Time_Sec Cycle_Time_Status
0   STN-01 2026-03-01 07:10:00             NaN            Normal
1   STN-05 2026-03-01 06:01:00             NaN            Normal
2   STN-01 2026-03-01 06:02:00         -4080.0          Too Fast
3   STN-05 2026-03-01 06:03:00           120.0            Normal
4   STN-03 2026-03-01 06:05:00             NaN            Normal
5   STN-02 2026-03-01 06:06:00             NaN            Normal
6   STN-03 2026-03-01 06:07:00           120.0            Normal
7   STN-04 2026-03-01 06:08:00             NaN            Normal
8   STN-05 2026-03-01 06:10:00           420.0          Too Slow
9   STN-03 2026-03-01 06:11:00           240.0            Normal
10  STN-03 2026-03-01 06:12:00            60.0            Normal
11  STN-05 2026-03-01 06:13:00           180.0            Normal
12  STN-05 2026-03-01 06:15:00           120.0            Normal
13  STN-03 2026-03-01 06:16:00           240.0            Normal
14  STN-04 2026-03-01 06:

In [8]:
production.to_csv(
    "production_events_cleaned_T11.csv",
    index=False
)

print("\nCycle Time Plausibility Check Completed Successfully!")


Cycle Time Plausibility Check Completed Successfully!


# Test Case 12

In [9]:
production = pd.read_csv("production_events_cleaned_T11.csv")
workorder = pd.read_csv("work_orders.csv")

In [10]:
production["Work_Order"] = production["Work_Order"].astype(str).str.strip().str.upper()
production["VIN"] = production["VIN"].astype(str).str.strip().str.upper()

workorder["Work_Order"] = workorder["Work_Order"].astype(str).str.strip().str.upper()
workorder["VIN"] = workorder["VIN"].astype(str).str.strip().str.upper()


In [11]:
production = production.merge(
    workorder[["Work_Order", "VIN"]],
    on="Work_Order",
    how="left",
    suffixes=("", "_Master")
)


In [12]:
production["WO_VIN_Status"] = np.where(
    production["VIN"] == production["VIN_Master"],
    "Valid",
    "Invalid"
)

In [13]:
print(production[
    [
        "Work_Order",
        "VIN",
        "VIN_Master",
        "WO_VIN_Status"
    ]
].head(20))


   Work_Order                VIN         VIN_Master WO_VIN_Status
0     WO10000            XX12234  MA1AB100000000000       Invalid
1     WO10001  MA1A**********001  MA1AB100000000001       Invalid
2     WO10002  MA1A**********002  MA1AB100000000002       Invalid
3     WO10003  MA1A**********003  MA1AB100000000003       Invalid
4     WO10004  MA1A**********004  MA1AB100000000004       Invalid
5     WO10005  MA1A**********005  MA1AB100000000005       Invalid
6     WO10006  MA1A**********006  MA1AB100000000006       Invalid
7     WO10007  MA1A**********007  MA1AB100000000007       Invalid
8     WO10008  MA1A**********008  MA1AB100000000008       Invalid
9     WO10009  MA1A**********009  MA1AB100000000009       Invalid
10    WO10010  MA1A**********010  MA1AB100000000010       Invalid
11    WO10011  MA1A**********011  MA1AB100000000011       Invalid
12    WO10012  MA1A**********012  MA1AB100000000012       Invalid
13    WO10013  MA1A**********013  MA1AB100000000013       Invalid
14    WO10

In [14]:
production.to_csv(
    "production_events_cleaned_T12.csv",
    index=False
)

print("\nWork Order Validation Completed Successfully!")


Work Order Validation Completed Successfully!


# Test Case 13

In [15]:
production = pd.read_csv("production_events_cleaned_T12.csv")

In [16]:
production["Timestamp"] = pd.to_datetime(
    production["Timestamp"],
    errors="coerce"
)


In [17]:
production["VIN"] = (
    production["VIN"]
    .astype(str)
    .str.strip()
    .str.upper()
)

production["Station"] = (
    production["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)


In [18]:
production = production.sort_values(
    by=["VIN", "Station", "Timestamp"]
)

production["Visit_No"] = (
    production.groupby(["VIN", "Station"])
    .cumcount() + 1
)


In [19]:
production["Rework_Status"] = production["Visit_No"].apply(
    lambda x: "Rework" if x > 1 else "Normal"
)


In [20]:
print(production[
    [
        "VIN",
        "Station",
        "Timestamp",
        "Visit_No",
        "Rework_Status"
    ]
].head(20))


                    VIN Station           Timestamp  Visit_No Rework_Status
1000  MA1A**********000  STN-03 2026-03-02 02:50:00         1        Normal
4000  MA1A**********000  STN-03 2026-03-04 17:20:00         2        Rework
2000  MA1A**********000  STN-04 2026-03-02 23:40:00         1        Normal
3000  MA1A**********000  STN-05 2026-03-03 20:30:00         1        Normal
2001  MA1A**********001  STN-01 2026-03-02 23:41:00         1        Normal
1001  MA1A**********001  STN-03 2026-03-02 02:51:00         1        Normal
3001  MA1A**********001  STN-04 2026-03-03 20:31:00         1        Normal
4001  MA1A**********001  STN-04 2026-03-04 17:21:00         2        Rework
1     MA1A**********001  STN-05 2026-03-01 06:01:00         1        Normal
2     MA1A**********002  STN-01 2026-03-01 06:02:00         1        Normal
1002  MA1A**********002  STN-01 2026-03-02 02:52:00         2        Rework
3002  MA1A**********002  STN-03 2026-03-03 20:32:00         1        Normal
4002  MA1A**

In [21]:
production.to_csv(
    "production_events_cleaned_13.csv",
    index=False
)

print("\nRework Loop Identification Completed Successfully!")


Rework Loop Identification Completed Successfully!


# Test Case 14

In [22]:
production = pd.read_csv("production_events_cleaned_13.csv")

In [23]:
production["Shift"] = (
    production["Shift"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [24]:
print(production["Timestamp"].dtype)

object


In [25]:
production["Timestamp"] = pd.to_datetime(
    production["Timestamp"],
    errors="coerce",
    format="mixed"
)

In [26]:
print(production["Timestamp"].dtype)

datetime64[ns]


In [27]:
def derive_shift(timestamp):

    timestamp = pd.to_datetime(timestamp, errors="coerce")

    if pd.isna(timestamp):
        return "UNKNOWN"

    hour = timestamp.hour

    if 6 <= hour < 14:
        return "A"
    elif 14 <= hour < 22:
        return "B"
    else:
        return "C"

In [28]:
production["Derived_Shift"] = production["Timestamp"].apply(derive_shift)

In [29]:
production["Shift_Status"] = np.where(
    production["Shift"] == production["Derived_Shift"],
    "Valid",
    "Invalid"
)

In [30]:
print(production[
    [
        "Timestamp",
        "Shift",
        "Derived_Shift",
        "Shift_Status"
    ]
].head(20))

             Timestamp Shift Derived_Shift Shift_Status
0  2026-03-02 02:50:00     B             C      Invalid
1  2026-03-04 17:20:00   NAN             B      Invalid
2  2026-03-02 23:40:00     C             C        Valid
3  2026-03-03 20:30:00   NAN             B      Invalid
4  2026-03-02 23:41:00   NAN             C      Invalid
5  2026-03-02 02:51:00     C             C        Valid
6  2026-03-03 20:31:00     A             B      Invalid
7  2026-03-04 17:21:00     B             B        Valid
8  2026-03-01 06:01:00   NAN             A      Invalid
9  2026-03-01 06:02:00     A             A        Valid
10 2026-03-02 02:52:00     B             C      Invalid
11 2026-03-03 20:32:00     A             B      Invalid
12 2026-03-04 17:22:00     C             B      Invalid
13 2026-03-02 23:42:00     B             C      Invalid
14 2026-03-02 02:53:00     A             C      Invalid
15 2026-03-02 23:43:00     A             C      Invalid
16 2026-03-04 17:23:00   NAN             B      

In [31]:
production.to_csv(
    "production_events_cleaned_14.csv",
    index=False
)

print("\nShift Code Derivation and Validation Completed Successfully!")


Shift Code Derivation and Validation Completed Successfully!


# Test Case 15

In [32]:
quality = pd.read_csv("quality_inspection_cleaned_T6.csv")

In [33]:
print(quality.columns.tolist())

['Inspection_ID', 'Event_ID', 'VIN', 'Defect_Code', 'Defect', 'Inspection_Source', 'Inspector_ID', 'Rework_Flag', 'Scrap_Reason', 'Result']


In [34]:

scrap_codes = ["SR01","SR02","SR03","SR04","SR05","SR06"]

quality["Scrap_Code"] = [
    random.choice(scrap_codes) if x=="REJECT" else None
    for x in quality["Defect"]
]

In [35]:
scrap_mapping = {
    "SR01":"Welding Defect",
    "SR02":"Paint Defect",
    "SR03":"Crack",
    "SR04":"Dimension Error",
    "SR05":"Material Defect",
    "SR06":"Electrical Failure"
}

quality["Scrap_Reason"] = quality["Scrap_Code"].map(scrap_mapping)

In [36]:
print("Before Mapping:")
print(quality["Scrap_Code"].head(10))

Before Mapping:
0    None
1    None
2    None
3    None
4    SR01
5    None
6    SR06
7    None
8    SR02
9    None
Name: Scrap_Code, dtype: object


In [37]:
quality.to_csv(
    "quality_inspection_cleaned_T15.csv",
    index=False
)

print("\nScrap Reason Code Mapping Completed Successfully!")


Scrap Reason Code Mapping Completed Successfully!


# Test Case 16

In [38]:
sensor = pd.read_csv("sensor_readings_cleaned_T10.csv")

In [39]:
def convert_torque(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    number = re.findall(r"\d+\.?\d*", value)

    if len(number) == 0:
        return np.nan

    number = float(number[0])

    if "LB-FT" in value or "LBFT" in value:
        return round(number * 1.35582,2)

    else:
        return number

In [40]:
def convert_temperature(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    number = re.findall(r"\d+\.?\d*", value)

    if len(number) == 0:
        return np.nan

    number = float(number[0])

    if "F" in value:
        return round((number-32)*5/9,2)

    else:
        return number


In [41]:
def convert_pressure(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    number = re.findall(r"\d+\.?\d*", value)

    if len(number) == 0:
        return np.nan

    number = float(number[0])

    if "PSI" in value:
        return round(number*0.0689476,2)

    else:
        return number

In [42]:
sensor["Torque_Nm"] = sensor["Torque"].apply(convert_torque)

sensor["Temperature_C"] = sensor["Temperature"].apply(convert_temperature)

sensor["Pressure_bar"] = sensor["Pressure"].apply(convert_pressure)

In [43]:
print(sensor[
[
"Torque",
"Torque_Nm",
"Temperature",
"Temperature_C",
"Pressure",
"Pressure_bar"
]
].head(20))

    Torque  Torque_Nm Temperature  Temperature_C Pressure  Pressure_bar
0      NaN        NaN         85C           85.0  4.5 bar          4.50
1     45.0       45.0        35 C           35.0  5.2 bar          5.20
2     60.0       60.0         85C           85.0   65 psi          4.48
3      NaN        NaN        35 C           35.0  4.5 bar          4.50
4      NaN        NaN       104 F           40.0  4.5 bar          4.50
5     50.0       50.0        35 C           35.0  4.5 bar          4.50
6      NaN        NaN         85C           85.0   65 psi          4.48
7     50.0       50.0       104 F           40.0   72 psi          4.96
8     50.0       50.0        35 C           35.0  5.2 bar          5.20
9      NaN        NaN        95 F           35.0  5.2 bar          5.20
10     NaN        NaN        35 C           35.0   72 psi          4.96
11     NaN        NaN         85C           85.0  4.5 bar          4.50
12    45.0       45.0       104 F           40.0   65 psi       

In [44]:
sensor.to_csv("sensor_readings_cleaned_T16.csv",index=False)

print("\nUnit Conversion Completed Successfully!")


Unit Conversion Completed Successfully!


# Test Case 17

In [45]:
sensor = pd.read_csv("sensor_readings_cleaned_T16.csv")

In [46]:
sensor["Torque"] = pd.to_numeric(sensor["Torque"], errors="coerce")
sensor["Temperature"] = pd.to_numeric(sensor["Temperature"], errors="coerce")
sensor["Pressure"] = pd.to_numeric(sensor["Pressure"], errors="coerce")

In [47]:
def detect_anomaly(row):

    if (
        row["Torque"] < 40 or row["Torque"] > 60 or
        row["Temperature"] < 70 or row["Temperature"] > 100 or
        row["Pressure"] < 4 or row["Pressure"] > 8
    ):
        return "Out of Spec"

    return "Normal"


In [48]:
sensor["Anomaly_Status"] = sensor.apply(
    detect_anomaly,
    axis=1
)

In [49]:
print(sensor[
[
    "Sensor_ID",
    "Torque",
    "Temperature",
    "Pressure",
    "Anomaly_Status"
]
].head(20))


   Sensor_ID  Torque  Temperature  Pressure Anomaly_Status
0    SEN-034     NaN          NaN       NaN         Normal
1    SEN-037    45.0          NaN       NaN         Normal
2    SEN-038    60.0          NaN       NaN         Normal
3    SEN-008     NaN          NaN       NaN         Normal
4    SEN-003     NaN          NaN       NaN         Normal
5    SEN-048    50.0          NaN       NaN         Normal
6    SEN-048     NaN          NaN       NaN         Normal
7    SEN-010    50.0          NaN       NaN         Normal
8    SEN-018    50.0          NaN       NaN         Normal
9    SEN-035     NaN          NaN       NaN         Normal
10   SEN-039     NaN          NaN       NaN         Normal
11   SEN-012     NaN          NaN       NaN         Normal
12   SEN-005    45.0          NaN       NaN         Normal
13   SEN-018    60.0          NaN       NaN         Normal
14   SEN-035    60.0          NaN       NaN         Normal
15   SEN-005     NaN          NaN       NaN         Norm

In [50]:
print("\nSummary:")
print(sensor["Anomaly_Status"].value_counts())


Summary:
Anomaly_Status
Normal    10000
Name: count, dtype: int64


In [51]:
sensor.to_csv(
    "sensor_readings_cleaned_T17.csv",
    index=False
)

print("\nAnomaly Detection Completed Successfully!")


Anomaly Detection Completed Successfully!


# test case 18

In [52]:
quality = pd.read_csv("quality_inspection_cleaned_T15.csv")

In [53]:
quality["Inspector_ID"] = (
    quality["Inspector_ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [54]:
def inspection_source(inspector):

    if inspector.startswith("EMP"):
        return "Human"

    elif inspector.startswith("CAM"):
        return "Automated"

    elif inspector.startswith("ROB"):
        return "Automated"

    elif inspector.startswith("AUTO"):
        return "Automated"

    else:
        return "Unknown"

In [55]:
quality["Inspection_Source"] = quality["Inspector_ID"].apply(
    inspection_source
)

In [56]:
print(quality[
[
    "Inspector_ID",
    "Inspection_Source"
]].head(20))

   Inspector_ID Inspection_Source
0         INS69           Unknown
1         INS83           Unknown
2         INS58           Unknown
3         INS83           Unknown
4         INS31           Unknown
5         INS22           Unknown
6         INS50           Unknown
7         INS48           Unknown
8         INS55           Unknown
9         INS66           Unknown
10        INS53           Unknown
11        INS14           Unknown
12        INS51           Unknown
13        INS95           Unknown
14        INS38           Unknown
15        INS84           Unknown
16        INS10           Unknown
17        INS23           Unknown
18        INS79           Unknown
19        INS99           Unknown


In [57]:
print("\nInspection Source Count:")
print(quality["Inspection_Source"].value_counts())


Inspection Source Count:
Inspection_Source
Unknown    4000
Name: count, dtype: int64


In [58]:
quality.to_csv(
    "quality_inspection_cleaned_T18.csv",
    index=False
)

print("\nHuman vs Automated Inspection Tagging Completed Successfully!")


Human vs Automated Inspection Tagging Completed Successfully!


# Test case 19

In [59]:
production = pd.read_csv("production_events_cleaned_14.csv")
tool_master = pd.read_csv("tool_master.csv")

In [60]:
production["Tool_ID"] = (
    production["Tool_ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

tool_master["Tool_ID"] = (
    tool_master["Tool_ID"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [61]:
production = pd.merge(
    production,
    tool_master,
    on="Tool_ID",
    how="left"
)

In [62]:
production["Tool_Name"] = production["Tool_Name"].fillna("Unknown Tool")
production["Tool_Type"] = production["Tool_Type"].fillna("Unknown Type")

In [63]:
print(production[
    ["Tool_ID", "Tool_Name", "Tool_Type"]
].head(10))

  Tool_ID Tool_Name    Tool_Type
0  TL-006    Tool_6   Nut Runner
1  TL-045   Tool_45  Welding Gun
2  TL-081   Tool_81   Nut Runner
3  TL-093   Tool_93   Nut Runner
4  TL-004    Tool_4   Press Tool
5  TL-058   Tool_58    Robot Arm
6  TL-061   Tool_61   Nut Runner
7  TL-068   Tool_68   Press Tool
8  TL-076   Tool_76  Welding Gun
9  TL-078   Tool_78   Press Tool


In [64]:
production.to_csv("production_events_cleaned_T19.csv", index=False)

print("Tool ID Mapping Completed Successfully!")

Tool ID Mapping Completed Successfully!


# Test Case 20

In [65]:
production = pd.read_csv("production_events_cleaned_T19.csv")
bom = pd.read_csv("bom_master.csv")

In [66]:
print(bom.columns.tolist())

['Part_Number', 'Part_Name', 'Category', 'BOM_Version', 'Effective_Date', 'Unit_Cost_USD', 'Supplier_Code', 'Status', 'Plant', 'Effective_From', 'Effective_To']


In [67]:
production["Timestamp"] = pd.to_datetime(
    production["Timestamp"],
    errors="coerce",
    format="mixed"
)

In [68]:
bom["Effective_From"] = pd.to_datetime(
    bom["Effective_From"],
    errors="coerce"
)

In [69]:
bom["Effective_To"] = pd.to_datetime(
    bom["Effective_To"],
    errors="coerce"
)

In [70]:
print(production.columns.tolist())


['Event_ID', 'VIN', 'Work_Order', 'Model', 'Line', 'Station', 'Timestamp', 'Part_Number', 'Supplier_Code', 'Operator_ID', 'Tool_ID', 'Shift', 'BOM_Status', 'VIN_Status', 'Supplier_Name', 'City', 'State', 'Country', 'Supplier_Type', 'Quality_Rating', 'On_Time_Delivery_%', 'PPM_Defects', 'Contact_Email', 'Supplier_Status', 'Cycle_Time_Sec', 'Cycle_Time_Status', 'VIN_Master', 'WO_VIN_Status', 'Visit_No', 'Rework_Status', 'Derived_Shift', 'Shift_Status', 'Tool_Name', 'Tool_Type', 'Plant', 'Calibration_Date', 'Next_Due_Date', 'Status', 'Accuracy_%', 'Manufacturer', 'Serial_Number']


In [71]:
production["Part_Number"] = (
    production["Part_Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [72]:
bom["Part_Number"] = (
    bom["Part_Number"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [73]:
# Merge only on Part_Number
production = production.merge(
    bom,
    on="Part_Number",
    how="left"
)

In [74]:
production["BOM_Status"] = np.where(
    (
        production["Timestamp"] >= production["Effective_From"]
    ) &
    (
        production["Timestamp"] <= production["Effective_To"]
    ),
    "Valid",
    "Invalid"
)

In [75]:
print(
    production[
        [
            "Part_Number",
            "Timestamp",
            "Effective_From",
            "Effective_To",
            "BOM_Status"
        ]
    ].head(20)
)

   Part_Number           Timestamp Effective_From Effective_To BOM_Status
0     ENG-1352 2026-03-02 02:50:00     2026-01-01   2026-06-30      Valid
1     ENG-1363 2026-03-04 17:20:00     2026-01-01   2026-06-30      Valid
2     ENG-1104 2026-03-02 23:40:00     2026-03-01   2026-08-28      Valid
3     ENG-1204 2026-03-03 20:30:00     2025-12-01   2026-05-30      Valid
4     ENG-1401 2026-03-02 23:41:00     2026-01-01   2026-06-30      Valid
5     ENG-1012 2026-03-02 02:51:00     2026-01-01   2026-06-30      Valid
6     ENG-1032 2026-03-03 20:31:00     2026-01-01   2026-06-30      Valid
7     ENG-1151 2026-03-04 17:21:00     2025-12-01   2026-05-30      Valid
8     ENG-1379 2026-03-01 06:01:00     2026-03-01   2026-08-28      Valid
9     ENG-1111 2026-03-01 06:02:00     2025-12-01   2026-05-30      Valid
10    ENG-1311 2026-03-02 02:52:00     2026-01-01   2026-06-30      Valid
11    ENG-1291 2026-03-03 20:32:00     2026-03-01   2026-08-28      Valid
12    ENG-1136 2026-03-04 17:22:00    

In [76]:
production.to_csv("production_events_cleaned_T20.csv", index=False)

print("BOM Effectivity Date Check Completed Successfully!")

BOM Effectivity Date Check Completed Successfully!
